# Final Assignment - Green Patent Detection: Advanced Agentic Workflow with QLoRA

## QLORA fine-tuning
The fine-tuning was done in AI-Lab, the code is displayed below (for reference only, NOT to be run).

### QLoRA Fine-Tuning of Mistral-7B
To specialize a large language model for green patent classification, I fine-tuned Mistral-7B-Instruct-v0.2 using QLoRA (Quantized Low-Rank Adaptation). This approach enables efficient domain adaptation while significantly reducing GPU memory requirements.

Noteable QLoRA parameters:
- Rank (r): 16
    - The rank determines the size of the low-rank update matrices inserted into the transformer layers. A higher rank increases the adapter’s capacity to learn task-specific patterns but also increases the number of trainable parameters.
    - 16 was chosen as a balanced trade off between adaptation capacity and parameter efficiency, providing sufficient task-specific learning ability.
- LoRA alpha: 32
    - Scaling factor applied to the LoRA updates. It controls the strength of the adapter’s contribution relative to the frozen base model.
    - Setting alpha to 32 (2 * rank) ensures the adapter has meaningful influence on the model’s output while keeping updates stable.
- LoRA dropout: 0.05
    - Dropout rate applied to the LoRA layers during training to improve generalization and reduce overfitting.

QLoRA allows training only a small set of injected low-rank matrices while keeping the base model frozen in 4-bit precision. This reduces resource requirements while maintaining strong adaptation capability.

Training was performed using the train_silver split from patents_50k_green.parquet.

Training Configuration:
- Learning rate: 2e-5
    - A relatively small learning rate is used to ensure stable adaptation of the LoRA layers without destabilizing the pretrained base model.
- Epochs: 1
    - The model is trained for a single epoch, meaning it sees the entire training dataset once. This choice was made due to computational constraints and to reduce training time while still allowing the model to adapt to the task.

Training loss decreased steadily and stabilized, indicating successful convergence of the LoRA adapter.

Only the LoRA adapter weights were saved, meaning I have created a lightweight domain-adapted classifier while keeping the base model unchanged.

#### run.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=finetuning_llm
#SBATCH --time=8:00:00
#SBATCH --output=outputs/myjob_%j.log
#SBATCH --gres=gpu:1
#SBATCH --mem=64G
#SBATCH --cpus-per-task=4

set -e

echo "🚀 Job started on $(hostname)"

# Activate uv environment properly
source ~/.bashrc

# Run using uv
uv run python finetuning.py

echo "✅ Job finished"

#### finetuning.py

In [ ]:
import os
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os

os.makedirs("model/qlora_mistral_patent", exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Data loading
import pandas as pd
from datasets import Dataset

# Load parquet
df = pd.read_parquet("patents_50k_green.parquet")

# Filter train split
train_silver = df[df["split"] == "train_silver"].copy()

# Keep only what we need
train_silver = train_silver[["text", "is_green_silver"]].dropna()

# Ensure integer labels
train_silver["label"] = train_silver["is_green_silver"].astype(int)

def make_prompt(text):
    return (
        "Task: Classify whether the following patent claim text is related to green technology.\n"
        "Output ONLY one character: 1 (green) or 0 (not green). No explanation.\n\n"
        f"{text}\n\n"
        "Answer:"
    )

train_silver["prompt"] = train_silver["text"].apply(make_prompt)
train_silver["target"] = train_silver["label"].astype(str)

# Causal LM training format
train_silver["full"] = train_silver["prompt"] + " " + train_silver["target"]

dataset = Dataset.from_pandas(
    train_silver[["full"]],
    preserve_index=False
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 2048  # limit for patent lenght

def tokenize(batch):
    return tokenizer(
        batch["full"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

tokenized = dataset.map(tokenize, batched=True, remove_columns=["full"])

# QLoRA (4-bit) model load
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

# LoRA config 
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="model/qlora_mistral_patent",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,   # effective batch size 16
    learning_rate=2e-5,
    num_train_epochs=1,               # start with 1 to validate pipeline
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()

# Save LoRA adapters only (small)
trainer.model.save_pretrained("model/qlora_mistral_patent/lora_adapters")
tokenizer.save_pretrained("model/qlora_mistral_patent/tokenizer")

print("✅ Done. Saved adapters to model/qlora_mistral_patent/lora_adapters")

## Agentic classification setup
This time I chose to do the classification using agentic setup in AI-Lab as well due to computational resource needs.

The script shown below implements a green patent classification pipeline using a QLoRA-adapted Mistral model within an agentic setup.

The base model (Mistral-7B-Instruct-v0.2) is loaded in 4-bit quantized form using bitsandbytes to reduce memory usage. A LoRA adapter trained on green patent classification is then loaded using:

judge_model = PeftModel.from_pretrained(base_model, ADAPTER_LOCAL)

Only the Judge agent uses the QLoRA adapter. The Advocate and Skeptic agents use the base quantized model without adapters. This isolates domain specialization to the final decision step.

Intermediate results are checkpointed periodically to prevent data loss.

### Agentic Structure
For each patent claim:
- Advocate Agent (base model): Reads only the claim text and produces a short argument for classifying the claim as green technology, along with a binary stance (0/1) and an argument strength level (weak/medium/strong) in structured JSON.
- Skeptic Agent (base model): Reads only the claim text and produces a short argument against green classification, also returning a binary stance (0/1) and strength (weak/medium/strong) in structured JSON.
- Judge Agent (QLoRA-adapted model): Receives the claim text plus the Advocate and Skeptic arguments and returns the final green label (llm_green_suggested = 0/1), confidence level (low/medium/high), and a one-sentence rationale in structured JSON.

### Output Consistency Handling
Because the agents are generative models, strict JSON output could not be guaranteed by prompting alone. Several safeguards are implemented:
- Schema validation (Pydantic): The output is validated against a strict schema. If invalid, the model retries once before marking the row as failed.
- Normalization: Escaped underscores (e.g., llm\_green_suggested) are corrected before parsing.
- Robust JSON extraction: A brace-scanning function extracts the first valid JSON object containing required fields.

These steps ensure that even if the agents sometimes generate free text, the final stored output is reliable and structured.

### run.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=langchain_agentic
#SBATCH --time=03:00:00
#SBATCH --output=outputs/langchain_agentic_%j.log
#SBATCH --gres=gpu:1
#SBATCH --mem=64G
#SBATCH --cpus-per-task=8

set -e

echo "🚀 Job started on $(hostname)"

cd $SLURM_SUBMIT_DIR
source ~/.bashrc
mkdir -p outputs

export HF_HUB_DISABLE_XET=1

# Point to files
export IN_PATH="hitl_green_100.csv"
export OUT_PATH="agentic_gold.csv"

uv run python agentic_llm.py

echo "✅ Job finished"

### agentic_llm.py

In [ ]:
import os
import re
import json
import csv
import pandas as pd
import torch
from typing import Literal
from pydantic import BaseModel, Field, ConfigDict, StrictInt

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

# Config
IN_PATH = os.environ.get("IN_PATH", "hitl_green_100.csv")
OUT_PATH = os.environ.get("OUT_PATH", "agentic_gold.csv")

BASE_MODEL = os.environ.get("BASE_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")
DTYPE = torch.float16

ADAPTER_LOCAL = os.environ.get(
    "ADAPTER_LOCAL",
    "/ceph/home/student.aau.dk/de63sv/agentic_green_patents/qlora_mistral_patent2/lora_adapters",
)

MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "600"))
CHECKPOINT_EVERY = int(os.environ.get("CHECKPOINT_EVERY", "1"))

# Schemas 
class GreenPatentClassification(BaseModel):
    model_config = ConfigDict(extra="forbid")

    llm_green_suggested: StrictInt = Field(description="0 or 1 (1 = green technology, 0 = not green)")
    llm_confidence: Literal["low", "medium", "high"] = Field(description='One of: "low", "medium", "high"')
    llm_rationale: str = Field(description="Brief rationale for the final decision")


class AgentArgument(BaseModel):
    """
    STRICT:
    - stance must be int 0 or 1
    - strength must be exactly weak/medium/strong (string)
    - extra keys forbidden
    """
    model_config = ConfigDict(extra="forbid")

    argument: str = Field(description="At most two concise sentences; no extra lines")
    stance: StrictInt = Field(description="Must be 0 or 1 (integer)")
    strength: Literal["weak", "medium", "strong"] = Field(description='One of: "weak", "medium", "strong"')

# Helpers
def normalize_llm_json_text(s: str) -> str:
    s = (s or "").strip()
    # Fix Markdown-escaped underscores in keys/strings: llm\_green\_suggested -> llm_green_suggested
    return s.replace("\\_", "_")


def clean_argument(s: str) -> str:
    """Single-line, compact argument."""
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s


def csv_sanitize(raw: str) -> str:
    """
    Sanitize text for CSV consumption without changing semantics.
    """
    s = raw or ""
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", s)
    return s


def extract_json_with_keys(s: str, required_keys: list[str]) -> str:
    """
    Extract first balanced JSON object that contains required keys.
    """
    s = normalize_llm_json_text(s)

    i = 0
    while True:
        start = s.find("{", i)
        if start == -1:
            raise ValueError("No JSON object with required keys found")

        depth = 0
        in_str = False
        esc = False
        saw_required = False

        for j in range(start, len(s)):
            ch = s[j]

            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
                continue

            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    obj = s[start : j + 1]
                    if all(k in obj for k in required_keys):
                        obj = re.sub(r",\s*}", "}", obj)  # remove trailing comma before }
                        return obj
                    i = j + 1
                    break

            if not saw_required and all(k in s[start : j + 1] for k in required_keys):
                saw_required = True

        else:
            # reached end without closing brace
            if saw_required and depth > 0 and (not in_str):
                obj = s[start:]
                obj = re.sub(r",\s*$", "", obj)         # remove trailing comma at end
                obj = obj + ("}" * depth)               # close braces
                obj = re.sub(r",\s*}", "}", obj)        # remove any ,}
                return obj

            raise ValueError("Unclosed JSON while searching for result object")


def extract_clean_json(raw: str, required_keys: list[str]) -> str:
    """
    Extract the first JSON object containing required keys and
    return it as compact one-line JSON.
    Returns "" if extraction or parsing fails.
    """
    try:
        raw = normalize_llm_json_text(raw)
        j = extract_json_with_keys(raw, required_keys)
        obj = json.loads(j)  # strict JSON parse
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return ""


def parse_agent_output_strict(raw: str) -> AgentArgument:
    """
    STRICT: no coercion, no repairs, no fallback values.
    - Must be valid JSON with correct types and allowed values.
    - Extra keys forbidden.
    """
    raw = normalize_llm_json_text(raw)
    j = extract_json_with_keys(raw, ['"argument"', '"stance"', '"strength"'])
    parsed = AgentArgument.model_validate_json(j)

    if parsed.stance not in (0, 1):
        raise ValueError(f"stance must be 0 or 1, got {parsed.stance}")

    parsed.argument = clean_argument(parsed.argument)
    return parsed


def parse_judge_output_strict(raw: str) -> GreenPatentClassification:
    """
    Judge parsing: strict schema, but we still extract the JSON object from surrounding text.
    (No value coercion.)
    """
    raw = normalize_llm_json_text(raw)
    try:
        return GreenPatentClassification.model_validate_json(raw)
    except Exception:
        j = extract_json_with_keys(raw, ['"llm_green_suggested"', '"llm_confidence"', '"llm_rationale"'])
        return GreenPatentClassification.model_validate_json(j)


# Load models
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb,
)

judge_model = PeftModel.from_pretrained(base_model, ADAPTER_LOCAL)
judge_model.eval()

print("Judge model type:", type(judge_model), flush=True)
print("PEFT config:", judge_model.peft_config, flush=True)


# Generation
@torch.inference_mode()
def generate_chat(
    model,
    system: str,
    user: str,
    temperature: float = 0.0,
) -> str:
    messages = [
        {"role": "system", "content": system.strip()},
        {"role": "user", "content": user.strip()},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = dict(
        **inputs,
        do_sample=(temperature > 0.0),
        top_p=1.0,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    if temperature > 0.0:
        gen_kwargs["temperature"] = temperature

    out = model.generate(**gen_kwargs)

    full = tokenizer.decode(out[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    return full[len(prompt_text) :].strip()

# Prompts 
ADV_SYSTEM = "You are The Advocate (Pro-Green)."
SKE_SYSTEM = "You are The Skeptic (Greenwashing Detector)."
JUDGE_SYSTEM = "You are a senior patent examiner."


def advocate_user(text: str) -> str:
    return f"""
You are Agent 1 (The Advocate). Argue FOR classifying the claim as green technology (CPC Y02-related).

Rules:
- Use ONLY the claim text; do not add outside knowledge.
- "argument": at most TWO concise sentences.
- "stance": MUST be an integer 0 or 1 (no other values).
- "strength": MUST be exactly one of "weak", "medium", "strong".
- "stance" must be 1 only if the claim itself clearly supports green.
- "stance" must be 0 if there is no reasonable green interpretation from the claim text.
- "strength": "strong" if explicit environmental mechanism, "medium" if plausible indirect efficiency, "weak" if speculative.

Claim:
{text}

Return ONLY one JSON object and NOTHING else:
{{"argument":"...","stance":0|1,"strength":"weak|medium|strong"}}
""".strip()


def skeptic_user(text: str) -> str:
    return f"""
You are Agent 2 (The Skeptic). Argue AGAINST classifying the claim as green technology (CPC Y02-related).

Rules:
- Use ONLY the claim text; do not add outside knowledge.
- "argument": at most TWO concise sentences.
- "stance": MUST be an integer 0 or 1 (no other values).
- "strength": MUST be exactly one of "weak", "medium", "strong".
- "stance" must be 0 if the claim does not clearly describe an environmental mechanism.
- "stance" must be 1 only if the environmental benefit is explicit in the claim.
- "strength": "strong" if clearly not green, "medium" if likely not green but minor efficiency angle, "weak" if borderline.

Claim:
{text}

Return ONLY one JSON object and NOTHING else:
{{"argument":"...","stance":0|1,"strength":"weak|medium|strong"}}
""".strip()


def judge_user(text: str, advocate_argument: str, skeptic_argument: str) -> str:
    return f"""
You are Agent 3 (The Judge). Weigh the Advocate and Skeptic arguments and decide the final green classification.

Decision rule:
- llm_green_suggested = 1 if the claim clearly describes a technical contribution that reduces environmental impact.
- Otherwise llm_green_suggested = 0.
- Confidence must be one of: low, medium, high.
- Rationale must be ONE sentence.

Claim:
{text}

Advocate argument:
{advocate_argument}

Skeptic argument:
{skeptic_argument}

Return ONLY one JSON object and NOTHING else:
{{"llm_green_suggested":0|1,"llm_confidence":"low|medium|high","llm_rationale":"..."}}
Do not escape underscores. Use field names exactly as shown.
""".strip()

# Pipeline
def ensure_output_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "advocate_raw",
        "skeptic_raw",
        "judge_raw",
        "advocate_argument",
        "advocate_stance",
        "advocate_strength",
        "advocate_error",
        "skeptic_argument",
        "skeptic_stance",
        "skeptic_strength",
        "skeptic_error",
        "llm_confidence",
        "llm_rationale",
        "llm_error",
    ]

    for col in cols:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype("object")

    if "llm_green_suggested" not in df.columns:
        df["llm_green_suggested"] = pd.Series([pd.NA] * len(df), dtype="Int64")
    else:
        df["llm_green_suggested"] = df["llm_green_suggested"].astype("Int64")

    return df


def run_agentic(text: str) -> dict:
    # Advocate
    advocate_raw_full = generate_chat(base_model, ADV_SYSTEM, advocate_user(text))

    # Save a one-line JSON if possible; otherwise save the full raw
    advocate_raw_saved = extract_clean_json(
        advocate_raw_full, ['"argument"', '"stance"', '"strength"']
    ) or advocate_raw_full

    advocate_argument = ""
    advocate_stance = ""
    advocate_strength = ""
    advocate_error = ""

    # Parse
    try:
        adv = parse_agent_output_strict(advocate_raw_full)
        advocate_argument = adv.argument
        advocate_stance = str(adv.stance)           # store as string column for consistency
        advocate_strength = adv.strength
    except Exception as e:
        advocate_error = f"{type(e).__name__}: {e}"

    # Skeptic
    skeptic_raw_full = generate_chat(base_model, SKE_SYSTEM, skeptic_user(text))
    skeptic_raw_saved = extract_clean_json(
        skeptic_raw_full, ['"argument"', '"stance"', '"strength"']
    ) or skeptic_raw_full

    skeptic_argument = ""
    skeptic_stance = ""
    skeptic_strength = ""
    skeptic_error = ""

    try:
        ske = parse_agent_output_strict(skeptic_raw_full)
        skeptic_argument = ske.argument
        skeptic_stance = str(ske.stance)
        skeptic_strength = ske.strength
    except Exception as e:
        skeptic_error = f"{type(e).__name__}: {e}"

    # Use parsed arguments if available; otherwise empty strings
    advocate_arg_for_judge = advocate_argument
    skeptic_arg_for_judge = skeptic_argument

    # Judge (retry, strict parse)
    last_err = None
    judge_raw_full = ""
    judge_raw_saved = ""
    for _ in range(2):
        judge_raw_full = generate_chat(
            judge_model,
            JUDGE_SYSTEM,
            judge_user(text, advocate_arg_for_judge, skeptic_arg_for_judge),
        )
        judge_raw_saved = extract_clean_json(
            judge_raw_full, ['"llm_green_suggested"', '"llm_confidence"', '"llm_rationale"']
        ) or judge_raw_full

        try:
            parsed = parse_judge_output_strict(judge_raw_full)

            if parsed.llm_green_suggested not in (0, 1):
                raise ValueError(f"llm_green_suggested must be 0/1, got {parsed.llm_green_suggested}")

            return {
                "advocate_raw": csv_sanitize(advocate_raw_saved),
                "skeptic_raw": csv_sanitize(skeptic_raw_saved),
                "judge_raw": csv_sanitize(judge_raw_saved),

                "advocate_argument": csv_sanitize(advocate_argument),
                "advocate_stance": advocate_stance,
                "advocate_strength": advocate_strength,
                "advocate_error": csv_sanitize(advocate_error),

                "skeptic_argument": csv_sanitize(skeptic_argument),
                "skeptic_stance": skeptic_stance,
                "skeptic_strength": skeptic_strength,
                "skeptic_error": csv_sanitize(skeptic_error),

                "llm_green_suggested": int(parsed.llm_green_suggested),
                "llm_confidence": parsed.llm_confidence,
                "llm_rationale": csv_sanitize(parsed.llm_rationale),
                "llm_error": "",
            }

        except Exception as e:
            last_err = e

    return {
        "advocate_raw": csv_sanitize(advocate_raw_saved),
        "skeptic_raw": csv_sanitize(skeptic_raw_saved),
        "judge_raw": csv_sanitize(judge_raw_saved),

        "advocate_argument": csv_sanitize(advocate_argument),
        "advocate_stance": advocate_stance,
        "advocate_strength": advocate_strength,
        "advocate_error": csv_sanitize(advocate_error),

        "skeptic_argument": csv_sanitize(skeptic_argument),
        "skeptic_stance": skeptic_stance,
        "skeptic_strength": skeptic_strength,
        "skeptic_error": csv_sanitize(skeptic_error),

        "llm_green_suggested": pd.NA,
        "llm_confidence": "",
        "llm_rationale": "",
        "llm_error": f"Judge invalid after retry. Last_err={type(last_err).__name__}: {last_err}",
    }


def main():
    df = pd.read_csv(IN_PATH)
    df = ensure_output_columns(df)

    n = len(df)
    for i, row in df.iterrows():
        print(f"Row {i+1}/{n}...", flush=True)
        text = str(row.get("text", ""))

        out = run_agentic(text)

        df.at[i, "advocate_raw"] = out.get("advocate_raw", "")
        df.at[i, "skeptic_raw"] = out.get("skeptic_raw", "")
        df.at[i, "judge_raw"] = out.get("judge_raw", "")

        df.at[i, "advocate_argument"] = out.get("advocate_argument", "")
        df.at[i, "advocate_stance"] = out.get("advocate_stance", "")
        df.at[i, "advocate_strength"] = out.get("advocate_strength", "")
        df.at[i, "advocate_error"] = out.get("advocate_error", "")

        df.at[i, "skeptic_argument"] = out.get("skeptic_argument", "")
        df.at[i, "skeptic_stance"] = out.get("skeptic_stance", "")
        df.at[i, "skeptic_strength"] = out.get("skeptic_strength", "")
        df.at[i, "skeptic_error"] = out.get("skeptic_error", "")

        val = out.get("llm_green_suggested", pd.NA)
        df.at[i, "llm_green_suggested"] = pd.NA if pd.isna(val) else int(val)

        df.at[i, "llm_confidence"] = out.get("llm_confidence", "")
        df.at[i, "llm_rationale"] = out.get("llm_rationale", "")
        df.at[i, "llm_error"] = out.get("llm_error", "")

        if (i + 1) % CHECKPOINT_EVERY == 0:
            df.to_csv(
                OUT_PATH,
                index=False,
                encoding="utf-8",
                quoting=csv.QUOTE_ALL,
                escapechar="\\",
                lineterminator="\n",
            )
            print(f"Checkpoint saved at row {i+1}", flush=True)

    df.to_csv(
        OUT_PATH,
        index=False,
        encoding="utf-8",
        quoting=csv.QUOTE_ALL,
        escapechar="\\",
        lineterminator="\n",
    )
    print("Saved ->", OUT_PATH, flush=True)


if __name__ == "__main__":
    main()

## Disagreement Reporting

### Agentic disagreement between the advocate and the skeptic
I had the advocate and skeptic output their stance (their 'vote') for each claim, where they were told to choose between 0 and 1. Out of 100 claims total, the agents disagreed on 60. This is a high level of disagreement, but it is relevant to note that theses patents are specifically chosen due to their uncertainty. 

In [1]:
import pandas as pd

df = pd.read_csv("agentic_gold.csv")

# Disagreement
df["agents_disagree"] = df["advocate_stance"].astype(str) != df["skeptic_stance"].astype(str)

# Low confidence
df["low_confidence"] = df["llm_confidence"].astype(str).str.lower() == "low"

# Needs human
df["needs_human"] = df["agents_disagree"] | df["low_confidence"]

# Gold label 
# Accept judge unless human review needed
df["is_green_gold"] = df["llm_green_suggested"].where(~df["needs_human"])

# Reporting
n_total = len(df)
n_disagree = int(df["agents_disagree"].sum())
n_lowconf = int(df["low_confidence"].sum())
n_human = int(df["needs_human"].sum())

print("="*60)
print(f"Total claims: {n_total}")
print(f"Agents disagreed on {n_disagree} out of {n_total} claims.")
print(f"Judge confidence was low on {n_lowconf} out of {n_total} claims.")
print(f"Total requiring human intervention: {n_human} out of {n_total} claims.")
print("="*60)

# Save HITL review file
hitl = df[df["needs_human"]].copy()
hitl.to_csv("hitl_review.csv", index=False)

print(f"Saved hitl_review.csv with {len(hitl)} rows for human review.")

Total claims: 100
Agents disagreed on 60 out of 100 claims.
Judge confidence was low on 45 out of 100 claims.
Total requiring human intervention: 97 out of 100 claims.
Saved hitl_review.csv with 97 rows for human review.


## HITL
I will now load the gold_100 dataset, which was classified by my agentic setup (prior to the HITL step) in this run. For a combined implementation of an agentic setup and HITL within the same pipeline, I refer to my Assignment 3 work using CrewAI and LangChain.

This run uses an exception-based HITL approach, meaning I do not manually review all 100 claims. Instead, I focus human review on:

- cases where there is disagreement between agents.
- cases where the Judge’s confidence is low.

However, the Judge flagged 45 claims as low-confidence (likely due to it being high uncertainty claims), which indicates that substantial human review is still required with the current setup.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

IN_PATH = "hitl_review.csv"
OUT_PATH = "hitl_review_labeled.csv"

hitl = pd.read_csv(IN_PATH)

# make sure columns exist
if "is_green_human" not in hitl.columns:
    hitl["is_green_human"] = np.nan
if "human_notes" not in hitl.columns:
    hitl["human_notes"] = ""

# columns to show
SHOW = [
    "doc_id",
    "llm_green_suggested", "llm_confidence", "llm_rationale",
    "advocate_stance", "advocate_strength", "advocate_argument",
    "skeptic_stance", "skeptic_strength", "skeptic_argument",
    "text",
]
SHOW = [c for c in SHOW if c in hitl.columns]

# start at first unlabeled
idx_list = list(hitl.index)
cur = 0
def is_labeled(v):
    s = str(v).strip()
    return s in ("0", "1")

while cur < len(idx_list) and is_labeled(hitl.at[idx_list[cur], "is_green_human"]):
    cur += 1

out = widgets.Output()
status = widgets.HTML()

btn_green = widgets.Button(description="1 = Green", button_style="success")
btn_not = widgets.Button(description="0 = Not green", button_style="danger")
btn_back = widgets.Button(description="Back")
btn_skip = widgets.Button(description="Skip")
btn_save = widgets.Button(description="Save", button_style="info")

notes = widgets.Textarea(description="Notes", layout=widgets.Layout(width="100%", height="90px"))

def save():
    hitl.to_csv(OUT_PATH, index=False)

def render():
    with out:
        clear_output()
        if cur < 0 or cur >= len(idx_list):
            print("Done.")
            return
        i = idx_list[cur]
        print(f"Row {cur+1}/{len(idx_list)}  (df index={i})")
        print("-" * 90)
        for c in SHOW:
            v = hitl.at[i, c]
            if pd.isna(v):
                v = ""
            print(f"\n[{c}]\n{v}")

        cur_label = str(hitl.at[i, "is_green_human"]).strip()
        if cur_label.lower() == "nan":
            cur_label = ""
        print("\n" + "-" * 90)
        print("Current is_green_human:", cur_label)

        notes.value = str(hitl.at[i, "human_notes"]) if not pd.isna(hitl.at[i, "human_notes"]) else ""

    status.value = f"<b>Saved to:</b> {OUT_PATH} &nbsp;&nbsp; <b>Labeled:</b> {sum(hitl['is_green_human'].astype(str).str.strip().isin(['0','1']))}/{len(hitl)}"

def next_row(step=1):
    global cur
    cur = min(len(idx_list) - 1, cur + step)

def on_green(_):
    global cur
    i = idx_list[cur]
    hitl.at[i, "is_green_human"] = 1
    hitl.at[i, "human_notes"] = notes.value.strip()
    save()
    next_row(1)
    render()

def on_not(_):
    global cur
    i = idx_list[cur]
    hitl.at[i, "is_green_human"] = 0
    hitl.at[i, "human_notes"] = notes.value.strip()
    save()
    next_row(1)
    render()

def on_skip(_):
    next_row(1)
    render()

def on_back(_):
    global cur
    cur = max(0, cur - 1)
    render()

def on_save(_):
    # save current notes even if not labeled
    if 0 <= cur < len(idx_list):
        i = idx_list[cur]
        hitl.at[i, "human_notes"] = notes.value.strip()
    save()
    render()

btn_green.on_click(on_green)
btn_not.on_click(on_not)
btn_skip.on_click(on_skip)
btn_back.on_click(on_back)
btn_save.on_click(on_save)

display(widgets.HBox([btn_green, btn_not, btn_back, btn_skip, btn_save]), notes, status, out)
render()

In [61]:
import pandas as pd
import numpy as np

full = pd.read_csv("agentic_gold.csv")
hitl = pd.read_csv("hitl_review_labeled.csv")

# merge human labels + notes back onto full
full = full.merge(
    hitl[["doc_id", "is_green_human", "human_notes"]],
    on="doc_id",
    how="left",
    suffixes=("", "_hitl"),
)

# prefer HITL columns if the full already had them
if "is_green_human_hitl" in full.columns:
    full["is_green_human"] = full["is_green_human_hitl"].combine_first(full.get("is_green_human"))
    full = full.drop(columns=["is_green_human_hitl"])

if "human_notes_hitl" in full.columns:
    full["human_notes"] = full["human_notes_hitl"].combine_first(full.get("human_notes"))
    full = full.drop(columns=["human_notes_hitl"])

# build is_green_gold:
# use human label if it is 0/1, else use llm_green_suggested
human = full["is_green_human"].astype(str).str.strip()
human_numeric = pd.to_numeric(human.where(human.isin(["0","1"])), errors="coerce")

full["is_green_gold"] = human_numeric.combine_first(full["llm_green_suggested"]).astype("Int64")

full.to_csv("agentic_gold_final.csv", index=False)
print("Saved -> agentic_gold_final.csv")

Saved -> agentic_gold_final.csv


/var/folders/dg/yxsfd9_j5n7bjns3tpn_bbfh0000gn/T/ipykernel_31113/3200372587.py:29: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  full["is_green_gold"] = human_numeric.combine_first(full["llm_green_suggested"]).astype("Int64")


## Fine-Tune PatentSBERTa

In [62]:
import pandas as pd
import numpy as np
import os, json, time, re

from typing import List, Optional, Dict

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import accelerate

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support

from langgraph.graph import StateGraph, END
from langgraph.types import interrupt  

from dotenv import load_dotenv
from tqdm.auto import tqdm
from groq import Groq
from pydantic import BaseModel, Field, ValidationError

from datasets import load_dataset
from datasets import Dataset, DatasetDict

In [ ]:
dfg = pd.read_csv('agentic_gold_final.csv') 
dfg.head() 

,doc_id,text,p_green,u,llm_green_suggested,llm_confidence,llm_rationale,is_green_human,human_notes,advocate_raw,...,advocate_argument,advocate_stance,advocate_strength,advocate_error,skeptic_argument,skeptic_stance,skeptic_strength,skeptic_error,llm_error,is_green_gold
0,9811268,1. A method for reducing disk read rate by man...,0.500037,0.999926,0,low,The claim does not explicitly describe a techn...,0.0,NaN,"{""argument"": ""This claim describes a method fo...",...,This claim describes a method for managing vir...,0,weak,NaN,This claim describes a method for managing vir...,0,weak,NaN,NaN,0
1,9371265,1. A prodrug of a JAK kinase inhibitor having ...,0.499629,0.999258,1,medium,Potential for increased bioavailability and re...,1.0,NaN,"{""argument"": ""This claim describes a prodrug o...",...,This claim describes a prodrug of a JAK kinase...,1,medium,NaN,This claim describes a prodrug formula without...,0,weak,NaN,NaN,1
2,9517793,1. A method of preventing over stroke in a rea...,0.499600,0.999201,0,low,The claim does not describe a technical contri...,0.0,NaN,"{""argument"": ""This claim describes a method fo...",...,This claim describes a method for preventing o...,0,weak,NaN,This claim describes a method for preventing o...,0,weak,NaN,NaN,0
3,9545611,1. A method of initiating chain reaction compr...,0.500424,0.999153,0,low,No explicit environmental contribution stated ...,0.0,NaN,"{""argument"": ""This claim describes a method fo...",...,This claim describes a method for initiating a...,0,weak,NaN,This claim describes a method for initiating a...,0,weak,NaN,NaN,0
4,9024233,1. A method of cleaning a side edge of a thin ...,0.500439,0.999123,1,medium,Reduces material usage and manufacturing costs...,1.0,NaN,"{""argument"": ""This method improves the efficie...",...,This method improves the efficiency of thin fi...,1,medium,NaN,This claim describes a method for cleaning the...,0,weak,NaN,NaN,1


In [64]:
# ensure numeric 0/1
ai = pd.to_numeric(dfg["llm_green_suggested"], errors="coerce")
human = pd.to_numeric(dfg["is_green_human"], errors="coerce")

agreement_pct = (ai == human).mean() * 100

print(f"Assignment 3 agreement: {agreement_pct:.2f}%")

Assignment 3 agreement: 97.00%


## Combining silver and gold labels

In [65]:
PARQUET_PATH = "patents_50k_green.parquet"
GOLD_CSV = "agentic_gold_final.csv"
OUT_PARQUET = "patents_50k_green_with_gold.parquet" 

df = pd.read_parquet(PARQUET_PATH)
gold = pd.read_csv(GOLD_CSV)

# normalize key column
if "id" not in gold.columns and "doc_id" in gold.columns:
    gold = gold.rename(columns={"doc_id": "id"})

# types
df["id"] = df["id"].astype(str)
gold["id"] = gold["id"].astype(str)

# human gold labels (100 only)
gold["is_green_gold"] = pd.to_numeric(gold["is_green_gold"], errors="coerce")
gold = gold.dropna(subset=["is_green_gold"]).copy()
gold["is_green_gold"] = gold["is_green_gold"].astype(int)

# merge
df = df.merge(gold[["id", "is_green_gold"]], on="id", how="left")

# rename for clarity
df = df.rename(columns={"is_green_gold": "is_green_gold_100"})

# combined label: gold_100 overrides silver
df["is_green_combined"] = df["is_green_silver"].astype(int)
mask_gold = df["is_green_gold_100"].notna()
df.loc[mask_gold, "is_green_combined"] = df.loc[mask_gold, "is_green_gold_100"].astype(int)

print("Gold_100 rows merged:", int(mask_gold.sum()))
print(
    df.loc[mask_gold, ["id", "is_green_silver", "is_green_gold_100", "is_green_combined"]]
      .head()
)

df.to_parquet(OUT_PARQUET, index=False)
print("Saved:", OUT_PARQUET)

Gold_100 rows merged: 100
           id  is_green_silver  is_green_gold_100  is_green_combined
129   8489858                1                1.0                  1
721   8527916                1                0.0                  0
761   9630890                1                1.0                  1
876   8471197                0                1.0                  1
1549  9447175                1                0.0                  0
Saved: patents_50k_green_with_gold.parquet


## Training data

In [66]:
df = pd.read_parquet("patents_50k_green_with_gold.parquet") 

# 1) base training split
train_silver = df[df["split"] == "train_silver"].copy()
train_silver["label"] = train_silver["is_green_combined"].astype(int)

# 2) gold_100 rows 
gold_100 = df[df["is_green_gold_100"].notna()].copy()
gold_100["label"] = gold_100["is_green_gold_100"].astype(int)

# 3) combine (append gold_100 to training)
train_final = pd.concat([train_silver, gold_100], ignore_index=True)

print("train_silver:", len(train_silver))
print("gold_100:", len(gold_100))
print("train_final:", len(train_final))

train_silver: 30000
gold_100: 100
train_final: 30100


## Evaluation data

In [67]:
eval_silver = df[df["split"] == "eval_silver"].copy()
eval_silver["label"] = eval_silver["is_green_silver"].astype(int)

eval_gold = gold_100.copy() 

print("eval_silver:", len(eval_silver))
print("eval_gold:", len(eval_gold))

eval_silver: 10000
eval_gold: 100


In [68]:
# keep only columns we need
train_final = train_final[["text", "label"]]
eval_silver = eval_silver[["text", "label"]]
gold_100_eval = gold_100[["text", "label"]]

## Model Training
The model was fine-tuned for one epoch using a maximum sequence length of 256 tokens and a learning rate of 2e-5, following the recommended settings to keep computation reasonable. Tokenization was performed using the PatentSBERTa tokenizer prior to training.

Model performance was evaluated on the held-out eval_silver split and gold_100 subset so evaluate performance, which will be used for a comparative analysis between baseline model, A2 model, A3 model and this final model.


In [ ]:
MODEL_NAME = "AI-Growth-Lab/PatentSBERTa"
MAX_LEN = 256      # According to assignment 2 details
LR = 2e-5          # Learning rate, within 1e-5 to 2e-5 recommended
EPOCHS = 1.        # According to assignment 2 details
BATCH = 16         # Can be lowered in case of problems with running
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Turn pandas dataframes into Hugging Face dataset
train_ds = Dataset.from_pandas(train_final, preserve_index=False)
eval_ds  = Dataset.from_pandas(eval_silver, preserve_index=False)
gold_ds  = Dataset.from_pandas(gold_100_eval, preserve_index=False)

# Tokenize before training using map function
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_ds  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])
gold_ds  = gold_ds.map(tokenize, batched=True, remove_columns=["text"])

train_ds.set_format("torch")
eval_ds.set_format("torch")
gold_ds.set_format("torch")

# Metrics for evaluation
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

# This controls training behavior
args = TrainingArguments(
    output_dir="patentsbberta_green_ft",
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    eval_strategy="epoch",
    save_strategy="no",      
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

# This runs the full fine-tuning loop
trainer.train()

In [ ]:
OUT_DIR = "patentsbberta_green_final"
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

In [ ]:
from huggingface_hub import HfApi, create_repo

OUT_DIR = "patentsbberta_green_final"
MODEL_REPO = "Signe22/patentsberta-green-hitl-final"

# create API
api = HfApi()

# create repo if it does not exist
create_repo(
    repo_id=MODEL_REPO,
    repo_type="model",
    exist_ok=True
)

# upload model files
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=MODEL_REPO,
    repo_type="model",
)

In [ ]:
from datasets import Dataset, DatasetDict

DATA_REPO = "Signe22/patents-50k-green-hitl-final"

df = pd.read_parquet("patents_50k_green_with_gold.parquet")

# Build splits
train_df = df[df["split"] == "train_silver"].copy()
eval_df  = df[df["split"] == "eval_silver"].copy()
pool_df  = df[df["split"] == "pool_unlabeled"].copy()

ds = DatasetDict({
    "train_silver": Dataset.from_pandas(train_df, preserve_index=False),
    "eval_silver": Dataset.from_pandas(eval_df, preserve_index=False),
    "pool_unlabeled": Dataset.from_pandas(pool_df, preserve_index=False),
})

# Push to HF Hub
ds.push_to_hub(DATA_REPO)

In [60]:
OUT_DIR = "patentsbberta_green_final"
model = AutoModelForSequenceClassification.from_pretrained(OUT_DIR)

trainer = Trainer(
    model=model,
    args=args,
    eval_dataset=eval_ds,         
    compute_metrics=compute_metrics,
)

print("=== Eval on eval_silver ===")
print(trainer.evaluate(eval_dataset=eval_ds))

print("=== Eval on gold_100 ===")
print(trainer.evaluate(eval_dataset=gold_ds))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

=== Eval on eval_silver ===


/opt/anaconda3/envs/NN-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.4206790030002594, 'eval_model_preparation_time': 0.0011, 'eval_accuracy': 0.8075, 'eval_precision': 0.8159851301115242, 'eval_recall': 0.7925777331995988, 'eval_f1': 0.8041111224178285, 'eval_runtime': 450.5752, 'eval_samples_per_second': 22.194, 'eval_steps_per_second': 1.387}
=== Eval on gold_100 ===


/opt/anaconda3/envs/NN-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.4960396885871887, 'eval_model_preparation_time': 0.0011, 'eval_accuracy': 0.74, 'eval_precision': 0.7636363636363637, 'eval_recall': 0.7636363636363637, 'eval_f1': 0.7636363636363637, 'eval_runtime': 5.4485, 'eval_samples_per_second': 18.354, 'eval_steps_per_second': 1.285}


## Comparative Analysis

### Model Performance Comparison

The table below compares model performance across the four stages of the project using the same silver evaluation set.

| Model Version | Training Data Source | F1 Score (Eval Set) |
|---------------|----------------------|---------------------|
| Baseline | Frozen PatentSBERTa embeddings (no fine-tuning) | 0.7719 |
| Assignment 2 Model | Fine-tuned on Silver + Gold (Simple LLM labels + HITL) | 0.8030 |
| Assignment 3 Model | Fine-tuned on Silver + Gold (Agentic LLM + HITL) | 0.8051 |
| Final Assignment Model | Fine-tuned on Silver + Gold (QLoRA-Powered MAS + Targeted HITL) | 0.8041 |

### Reflection
The final QLoRA-powered Multi-Agent System (MAS) model achieved an F1 score of 0.8041 on the independent evaluation set, which is marginally lower than the Assignment 3 model (0.8051) but higher than earlier versions. The difference (0.001) is small, suggesting that the fine-tuned models perform at a comparable level on unseen data.

Although the final model achieved the highest F1 score on the Gold 100 subset, this dataset was also included during fine-tuning and therefore cannot be considered an independent evaluation benchmark. While the improved performance likely indicates stronger alignment with the high-quality gold annotations, it may also partially reflect closer fitting to examples seen during training. As such, results on this subset should be interpreted as evidence of improved calibration and label alignment rather than definitive proof of improved generalization.

Overall, the results indicate that the QLoRA-powered MAS with targeted HITL improved alignment with human judgments while maintaining stable performance on the broader evaluation set.